# Project Setup

In this notebook we will setup the project.

## Download the dataset

Run the cell below to download the dataset (~24 GB) and print the path to the download.

### Dataset File Structure

More information can be found in the README for the dataset but here is a short summary/explanation.

DIRECTORY STRUCTURE

- ./ASVspoof2019_root
  - LA (logical access)
    - ASVspoof2019_LA_asv_protocols
    - ASVspoof2019_LA_asv_scores
    - ASVspoof2019_LA_cm_protocols (labels for the audios)
    - ASVspoof2019_LA_dev (contains flac folder of audios)
    - ASVspoof2019_LA_eval (contains flac folder of audios)
    - ASVspoof2019_LA_train(contains flac folder of audios) 
    - README.LA.txt
  - PA (physical access)
    - ASVspoof2019_PA_asv_protocols
    - ASVspoof2019_PA_asv_scores
    - ASVspoof2019_PA_cm_protocols (labels for the audios)
    - ASVspoof2019_PA_dev (contains flac folder of audios)
    - ASVspoof2019_PA_eval (contains flac folder of audios)
    - ASVspoof2019_PA_train (contains flac folder of audios)
    - README.PA.txt
  - asvspoof2019_evaluation_plan.pdf
  - asvspoof2019_Interspeech2019_submission.pdf
  - README.txt

In [1]:
import sys
!{sys.executable} -m pip install --user kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")

print("Path to dataset files:", path)

/home1/ihsiao/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home1/ihsiao/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1


## Manifest Creation

Next we will create manifests to organize our data. Run the cells below for this.

In [2]:
from pathlib import Path
import pandas as pd
import csv

# This should already exist from kagglehub
DATASET_ROOT = Path(path)

# Hardcode LA structure based on what you showed
LA_ROOT = DATASET_ROOT / "LA" / "LA"

train_audio = LA_ROOT / "ASVspoof2019_LA_train" / "flac"
dev_audio   = LA_ROOT / "ASVspoof2019_LA_dev" / "flac"
eval_audio  = LA_ROOT / "ASVspoof2019_LA_eval" / "flac"

protocol_dir = LA_ROOT / "ASVspoof2019_LA_cm_protocols"

train_protocol = protocol_dir / "ASVspoof2019.LA.cm.train.trn.txt"
dev_protocol   = protocol_dir / "ASVspoof2019.LA.cm.dev.trl.txt"
eval_protocol  = protocol_dir / "ASVspoof2019.LA.cm.eval.trl.txt"

print(train_audio.exists(), train_protocol.exists())


True True


In [3]:
from pathlib import Path
import csv

def build_manifest_robust(protocol_path: Path, audio_dir: Path, out_csv: Path):
    """
    Robust manifest builder for ASVspoof protocol files where lines look like:
      <spk_id> <utt_id> <...> <label>
    We take utt_id = parts[1], label = parts[-1] (bonafide/spoof).
    """
    protocol_path = Path(protocol_path)
    audio_dir = Path(audio_dir)
    audio_map = {p.stem: p for p in audio_dir.rglob("*.flac")}
    rows = []
    missing = []
    with open(protocol_path, "r", errors="ignore") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 2:
                print(f"Skipping malformed line {i}: {line}")
                continue
            utt = parts[1]
            label_token = parts[-1].lower()
            label = 0 if "bonafide" in label_token or "genuine" in label_token else 1
            wav_path = audio_map.get(utt)
            if wav_path is None:
                # try with common alternative stems (some files may have suffixes)
                # e.g., utt might be 'LA_D_1047731' but files could be named 'LA_D_1047731-1' etc.
                alt = None
                for k in audio_map.keys():
                    if k.startswith(utt):
                        alt = audio_map[k]
                        break
                if alt is not None:
                    wav_path = alt
                else:
                    missing.append(utt)
                    continue
            rows.append((utt, str(wav_path.resolve()), label))

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with open(out_csv, "w", newline="") as outf:
        writer = csv.writer(outf)
        writer.writerow(["utt", "wav", "label"])
        for r in rows:
            writer.writerow(r)

    print(f"Wrote manifest {out_csv} with {len(rows)} entries.")
    if missing:
        print(f"Warning: {len(missing)} utt ids from protocol not found in {audio_dir}. Sample missing: {missing[:10]}")
    return out_csv


In [4]:
PROJECT_ROOT = Path.cwd()
MANIFESTS = PROJECT_ROOT / "manifests"
MANIFESTS.mkdir(parents=True, exist_ok=True)

build_manifest_robust(train_protocol, train_audio, MANIFESTS / "LA_train.csv")
build_manifest_robust(dev_protocol, dev_audio, MANIFESTS / "LA_dev.csv")
build_manifest_robust(eval_protocol, eval_audio, MANIFESTS / "LA_eval.csv")

Wrote manifest /scratch1/ihsiao/Anti-Spoofing-Generalization/manifests/LA_train.csv with 25380 entries.
Wrote manifest /scratch1/ihsiao/Anti-Spoofing-Generalization/manifests/LA_dev.csv with 24844 entries.
Wrote manifest /scratch1/ihsiao/Anti-Spoofing-Generalization/manifests/LA_eval.csv with 71237 entries.


PosixPath('/scratch1/ihsiao/Anti-Spoofing-Generalization/manifests/LA_eval.csv')

### Quick Sanity Check

LA_train.csv 25380

Bonafide: 2580

Spoof: 22800

LA_dev.csv 24844

Bonafide: 2548

Spoof: 22296

LA_eval.csv 71237

Bonafide: 7355

Spoof: 63882

In [5]:
import pandas as pd

for split in ["LA_train.csv", "LA_dev.csv", "LA_eval.csv"]:
    df = pd.read_csv(MANIFESTS / split)
    print(split, len(df))
    print("Bonafide:", (df.label == 0).sum())
    print("Spoof:", (df.label == 1).sum())
    print()


LA_train.csv 25380
Bonafide: 2580
Spoof: 22800

LA_dev.csv 24844
Bonafide: 2548
Spoof: 22296

LA_eval.csv 71237
Bonafide: 7355
Spoof: 63882

